# ADL, ADQ, Bayes ingenuo y k vecinos (solucion)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.datasets import load_wine


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_NOTEBOOKS = "https://github.com/AdanReyes2407/Notebooks_NM_ML.git"


def is_colab():
    return "google.colab" in sys.modules


def ensure_repo_clone(repo_url=REPO_NOTEBOOKS, target=Path('/content/Notebooks_NM_ML')):
    if is_colab() and not target.exists():
        print(f"Clonando repositorio en {target} ...")
        subprocess.run(["git", "clone", repo_url, str(target)], check=True)


def resolve_data_dir():
    ensure_repo_clone()
    candidates = [
        Path('/content/Notebooks_NM_ML/Bases de datos'),
        Path.cwd() / 'Bases de datos',
        Path.cwd().parent / 'Bases de datos',
        Path('/content/Bases de datos'),
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


In [ ]:
def load_dataset(data_dir):
    if data_dir is not None:
        p = data_dir / 'column_2C.dat'
        if p.exists():
            df = pd.read_csv(p, delim_whitespace=True, header=None)
            X = df.iloc[:, :6].copy()
            y_raw = df.iloc[:, 6].astype(str).str.upper().str.strip()
            y = (y_raw == 'ABNORMAL').astype(int)
            return X, y, 'column_2C.dat'

    wine = load_wine(as_frame=True)
    X = wine.data.copy()
    y = wine.target.astype(int).copy()
    return X, y, 'sklearn_wine'


DATA_DIR = resolve_data_dir()
X, y, fuente = load_dataset(DATA_DIR)
print('Fuente:', fuente)
print('X shape:', X.shape)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


In [ ]:
models = {
    'ADL (LDA)': LinearDiscriminantAnalysis(),
    'ADQ (QDA)': QuadraticDiscriminantAnalysis(),
    'Bayes ingenuo': GaussianNB(),
    'k-NN (k=7)': KNeighborsClassifier(n_neighbors=7),
}

rows = []
preds = {}
for name, m in models.items():
    m.fit(X_train_s, y_train)
    y_hat = m.predict(X_test_s)
    preds[name] = y_hat
    rows.append((name, accuracy_score(y_test, y_hat)))

res = pd.DataFrame(rows, columns=['Modelo', 'Accuracy']).sort_values('Accuracy', ascending=False)
print(res)


In [ ]:
best = res.iloc[0, 0]
print('Mejor modelo:', best)
print(classification_report(y_test, preds[best], digits=4))

fig, ax = plt.subplots(figsize=(5.2, 4.4))
ConfusionMatrixDisplay.from_predictions(y_test, preds[best], ax=ax, cmap='Blues')
ax.set_title(f'Matriz de confusion - {best}')
plt.tight_layout()
plt.show()


In [ ]:
k_vals = list(range(1, 26))
acc_vals = []
for k in k_vals:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    acc_vals.append(accuracy_score(y_test, knn.predict(X_test_s)))

plt.figure(figsize=(7.2, 4.1))
plt.plot(k_vals, acc_vals, marker='o')
plt.xlabel('k')
plt.ylabel('Accuracy test')
plt.title('Sensibilidad de k en k-NN')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()
